In [1]:
%pip install requests pandas beautifulsoup4 pybaseball

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cffi-2.0.0-cp311-cp311-macosx_10_13_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 49.5 MB/s  0:00:00m0:00:01
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import requests
import pandas as pd
import pybaseball as pb
import time
import re
from pathlib import Path
from bs4 import BeautifulSoup
import random

random.seed(42) 
VIDEO_TYPE = "HOME"                 # "HOME" or "AWAY" camera angle
START_DT = "2025-09-01"             # for now, just september
END_DT = "2025-09-28"
NUM_SAMPLED = 50
OUT_DIR = Path("../data/videos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/537.36"
})

print(f"Output directory: {OUT_DIR.resolve()}")


Output directory: /Users/apple/Documents/BU/Spring2026/CS585/Final-Project/Release-Point/data/videos


In [3]:
# get statcast data for 2025 season, need playid
df = pb.statcast(START_DT, END_DT)
# possibly need to filter to 
sampled_pitches = df.sample(n=NUM_SAMPLED, random_state=42).copy()

sampled_pitches.head()

This is a large query, it may take a moment to complete


100%|██████████| 28/28 [00:01<00:00, 17.96it/s]


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
116,SI,2025-09-06,94.4,-2.4,5.28,"Lord, Brad",673548,695418,single,hit_into_play,...,1,2.13,1.52,1.52,16.1,1.508801,-0.705946,35.734631,35.108287,30.622329
114,FF,2025-09-04,92.9,1.15,6.21,"Cameron, Noah",545361,702070,NaN,foul,...,1,1.29,0.54,-0.54,57.0,5.559896,6.13396,31.157565,30.81179,18.573191
2075,SL,2025-09-28,86.5,2.12,5.54,"Soto, Gregory",691594,642397,hit_by_pitch,hit_by_pitch,...,<NA>,3.14,-0.9,0.9,41.0,<NA>,<NA>,<NA>,<NA>,<NA>
3168,ST,2025-09-19,81.8,3.14,5.3,"Raley, Brooks",695734,548384,field_out,hit_into_play,...,1,2.95,-1.8,-1.8,28.1,11.755782,-41.636119,40.165454,43.240845,36.415389
2050,SI,2025-09-18,97.2,0.2,6.53,"Ashby, Aaron",694203,676879,NaN,ball,...,1,1.73,1.34,-1.34,49.1,<NA>,<NA>,<NA>,<NA>,<NA>


In [4]:
# make lookup for play_id and add to the CSV
def get_lookup_for_game(game_pk):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    feed = requests.get(url).json()

    rows = []
    for play in feed["liveData"]["plays"]["allPlays"]:
        ab = play["about"]["atBatIndex"] + 1            # API is 0-based; statcast at_bat_number is 1-based
        for event in play["playEvents"]:
            if event.get("isPitch"):
                rows.append({
                    "game_pk": game_pk,
                    "at_bat_number": ab,
                    "pitch_number": event["pitchNumber"],
                    "play_id": event["playId"],
                })
    return pd.DataFrame(rows)

game_pks = sampled_pitches["game_pk"].unique()
lookup_df = pd.concat([get_lookup_for_game(pk) for pk in game_pks], ignore_index=True)

sampled_pitches = sampled_pitches.merge(
    lookup_df,
    on=["game_pk", "at_bat_number", "pitch_number"],
    how="left"
)

print(sampled_pitches["play_id"])

missing = sampled_pitches["play_id"].isna().sum()
print(f"Merged: {len(sampled_pitches)} rows | Missing play_id: {missing}")

0     be6c4fa5-d3dd-3d70-956d-fd470efe8630
1     a6795e29-f12e-32c3-aa0e-f06ac75a772f
2     7f042c3a-f894-308f-b201-fcacb42e62cd
3     7162d643-fd10-3f31-b9ed-a117acd8792e
4     2662a7aa-c1c4-350e-8367-d53bed0a266f
5     97c2b3d0-340d-3a60-92af-d575023a29ed
6     56e30c6e-f8b9-37db-bbd6-b3205242f233
7     ff20a37e-9aa1-35db-bf9a-568335c7aaca
8     eaeffe7a-9ef3-35c8-a073-07e89ea1006f
9     1ac078b1-b675-3194-8e7a-8a3a97feb49b
10    0425c0c0-9ede-3cdc-8dbe-511a37adc646
11    8e8b0b8c-2f2d-3321-8052-2b00bf430bce
12    2c86aa9d-7eaa-3838-9a22-dfb534fdb94a
13    dba945a9-737f-3869-8b60-a134ab5f7312
14    62e20169-4558-32f6-8371-690ea8328fe9
15    e9361172-679e-39a7-98cf-f00df0f243ac
16    e923e047-fdb3-3c04-98de-eb07cac8fbc7
17    ce8338c8-7600-3b81-bde9-8d9be3487d76
18    c977bb3b-d56f-3560-a62d-6a562d4a8394
19    5399cd45-48f4-3b66-a579-649737b803a6
20    14e37386-9e15-324e-9211-600c63054c66
21    9dfc0de3-5873-392b-af52-b1d0a19336cb
22    4813abab-d1d2-3030-af58-655d16326ca9
23    561e2

In [6]:
# go to savant page and find mp4 link
def resolve_video_url(play_id: str, video_type: str = "HOME") -> str | None:
    page = SESSION.get(
        f"https://baseballsavant.mlb.com/sporty-videos?playId={play_id}", timeout=30
    )
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "html.parser")
    for source in soup.find_all("source"):
        src = source.get("src", "")
        if video_type.upper() in src.upper():
            return src
    first = soup.find("source", src=True)
    return first["src"] if first else None


def download_video(play_id: str, out_path: Path, video_type: str = "HOME") -> bool:
    video_url = resolve_video_url(play_id, video_type)
    if not video_url:
        return False
    r = SESSION.get(video_url, timeout=90, stream=True)
    r.raise_for_status()
    with out_path.open("wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
    return True


downloaded, failed = [], []

for _, row in sampled_pitches.iterrows():
    safe_type = row["pitch_type"] if row["pitch_type"] else "XX"
    fname     = f"{row['game_pk']}_{row['play_id'][:8]}_{safe_type}.mp4"
    out_path  = OUT_DIR / fname

    if out_path.exists():
        print(f"  [skip] {fname}")
        downloaded.append(row["play_id"])
        continue

    try:
        ok = download_video(row["play_id"], out_path, VIDEO_TYPE)
        if ok:
            kb = out_path.stat().st_size // 1024
            print(f"  [OK]   {fname}  ({kb} KB)  — {row['pitcher']} vs {row['batter']}")
            downloaded.append(row["play_id"])
        else:
            print(f"  [miss] {row['play_id']} — video URL not found")
            failed.append(row["play_id"])
    except Exception as e:
        print(f"  [ERR]  {row['play_id']}: {e}")
        failed.append(row["play_id"])

    time.sleep(1.2)

print(f"\nDone.  Downloaded: {len(downloaded)}  |  Failed / missing: {len(failed)}")


  [OK]   776440_be6c4fa5_SI.mp4  (7982 KB)  — 695418 vs 673548
  [OK]   776463_a6795e29_FF.mp4  (3928 KB)  — 702070 vs 545361
  [OK]   776148_7f042c3a_SL.mp4  (7294 KB)  — 642397 vs 691594
  [OK]   776259_7162d643_ST.mp4  (5457 KB)  — 548384 vs 695734
  [OK]   776274_2662a7aa_SI.mp4  (4672 KB)  — 676879 vs 694203
  [OK]   776262_97c2b3d0_FF.mp4  (6911 KB)  — 686613 vs 647304
  [OK]   776400_56e30c6e_SL.mp4  (3491 KB)  — 571945 vs 645302
  [OK]   776324_ff20a37e_ST.mp4  (5626 KB)  — 641927 vs 606466
  [OK]   776490_eaeffe7a_FF.mp4  (4718 KB)  — 643361 vs 690993
  [OK]   776306_1ac078b1_FF.mp4  (4495 KB)  — 641302 vs 702616
  [OK]   776448_0425c0c0_CH.mp4  (4288 KB)  — 669711 vs 682998
  [OK]   776321_8e8b0b8c_SI.mp4  (4942 KB)  — 665871 vs 677592
  [OK]   776503_2c86aa9d_FF.mp4  (4227 KB)  — 605135 vs 663647
  [OK]   776213_dba945a9_CH.mp4  (4694 KB)  — 694297 vs 518692
  [OK]   776425_62e20169_FF.mp4  (3843 KB)  — 685126 vs 668670
  [OK]   776462_e9361172_CU.mp4  (4918 KB)  — 671737 vs

In [7]:
# for metadata
sampled_pitches["downloaded"] = sampled_pitches["play_id"].isin(downloaded)
sampled_pitches["filename"] = sampled_pitches.apply(
    lambda r: f"{r['game_pk']}_{r['play_id'][:8]}_{r['pitch_type'] or 'XX'}.mp4",
    axis=1,
)

csv_path = OUT_DIR / "metadata.csv"
sampled_pitches.to_csv(csv_path, index=False)
print(f"Metadata saved → {csv_path}")
sampled_pitches.head()


Metadata saved → ../data/videos/metadata.csv


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,play_id,downloaded,filename
0,SI,2025-09-06,94.4,-2.4,5.28,"Lord, Brad",673548,695418,single,hit_into_play,...,1.52,16.1,1.508801,-0.705946,35.734631,35.108287,30.622329,be6c4fa5-d3dd-3d70-956d-fd470efe8630,True,776440_be6c4fa5_SI.mp4
1,FF,2025-09-04,92.9,1.15,6.21,"Cameron, Noah",545361,702070,NaN,foul,...,-0.54,57.0,5.559896,6.13396,31.157565,30.81179,18.573191,a6795e29-f12e-32c3-aa0e-f06ac75a772f,True,776463_a6795e29_FF.mp4
2,SL,2025-09-28,86.5,2.12,5.54,"Soto, Gregory",691594,642397,hit_by_pitch,hit_by_pitch,...,0.9,41.0,<NA>,<NA>,<NA>,<NA>,<NA>,7f042c3a-f894-308f-b201-fcacb42e62cd,True,776148_7f042c3a_SL.mp4
3,ST,2025-09-19,81.8,3.14,5.3,"Raley, Brooks",695734,548384,field_out,hit_into_play,...,-1.8,28.1,11.755782,-41.636119,40.165454,43.240845,36.415389,7162d643-fd10-3f31-b9ed-a117acd8792e,True,776259_7162d643_ST.mp4
4,SI,2025-09-18,97.2,0.2,6.53,"Ashby, Aaron",694203,676879,NaN,ball,...,-1.34,49.1,<NA>,<NA>,<NA>,<NA>,<NA>,2662a7aa-c1c4-350e-8367-d53bed0a266f,True,776274_2662a7aa_SI.mp4
